# Lab 1 — Tokenization and a smoothed bigram model

COMP-672 · John Glossner · 2 points

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week01_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab01_YourName.ipynb`, and work in that copy. 

[Course lab index and submission instructions](https://github.com/glossner/COMP-672-Labs/blob/main/README.md) · [Handout](https://github.com/glossner/COMP-672-Labs/blob/main/labs/LAB01.md)


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. 


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 1 — Tokenization and a smoothed bigram model


## Implementation tasks

1. Implement NFC-aware tokenization; document how punctuation, underscores, digits, and combining marks behave.

2. Implement weighted pair counting and deterministic BPE merging. Inspect at least six learned merges and segment three held-out words.

3. Implement normalized add-alpha bigram probabilities with a training-only vocabulary and explicit BOS, EOS, and UNK conventions.

4. Implement perplexity with token-weighted log losses; tune alpha on development documents using the provided candidate values.


Functions to complete: `tokenize`, `learn_bpe`, `Bigram.probability`, `Bigram.perplexity`.

## Required experiments

- Report development perplexity for alpha .01, .1, and 1; choose one before reporting the test score.

- Compare word, character, and BPE token counts on the same five sentences. Do not compare their raw per-token perplexities as if the units were identical.

## Brief analysis in your notebook or code comments

1. Explain why a distribution over successors must sum to one even for an unseen history.

2. Show a tokenizer failure case and propose a task-dependent remedy.

3. Explain what the generated corpus permits you to conclude and what a real-domain evaluation would add.


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.




## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 1 student starter. Original instructional corpus."""


In [ ]:
from collections import Counter, defaultdict


In [ ]:
import math


In [ ]:
import re


In [ ]:
import unicodedata


In [ ]:
import numpy as np


In [ ]:
def tokenize(text):
    """NFC, lower case, Unicode word sequences and individual punctuation."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def learn_bpe(words, n_merges=12):
    """Word->count input; character initialization; no word boundary symbol.
    Return an ordered list of merged pairs. Use lexical tie breaking.
    """
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def apply_bpe(word, merges):
    symbols=list(word)
    for a,b in merges:
        out=[]; i=0
        while i<len(symbols):
            if i+1<len(symbols) and symbols[i]==a and symbols[i+1]==b:
                out.append(a+b); i+=2
            else: out.append(symbols[i]); i+=1
        symbols=out
    return symbols


In [ ]:
def corpus():
    subjects=['robot','student','engineer','teacher','doctor','pilot','artist','researcher']
    verbs=['reads','checks','finds','moves','tests','describes']
    objects=['a signal','the report','a token','the sample']
    texts=[f'the {s} {v} {o} .' for s in subjects for v in verbs for o in objects]
    rng=np.random.default_rng(17); rng.shuffle(texts)
    return texts[:120],texts[120:156],texts[156:]


In [ ]:
class Bigram:
    def __init__(self, train, alpha=1.0):
        if alpha<=0: raise ValueError('alpha must be positive')
        self.vocab=sorted(set(t for s in train for t in tokenize(s))|{'<UNK>','<EOS>'})
        self.vset=set(self.vocab); self.alpha=alpha
        self.counts=Counter(); self.hist=Counter()
        for text in train:
            seq=['<BOS>']+tokenize(text)+['<EOS>']
            for a,b in zip(seq,seq[1:]): self.counts[a,b]+=1; self.hist[a]+=1
    def probability(self, history, token):
        raise NotImplementedError("Complete this laboratory task.")
    def perplexity(self, texts):
        """Include EOS, exclude BOS from scored-event count; map OOV to UNK."""
        raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def checks():
    assert tokenize('café')==tokenize('cafe\u0301')
    assert learn_bpe({'low':2,'lower':1},2)==[('l','o'),('lo','w')]
    m=Bigram(['a b','a c'])
    for h in ['a','<BOS>','unseen']:
        assert np.isclose(sum(m.probability(h,t) for t in m.vocab),1)
    assert math.isfinite(m.perplexity(['new unseen words']))


In [ ]:
def experiment():
    train,dev,test=corpus()
    results={a:Bigram(train,a).perplexity(dev) for a in [.01,.1,1.]}
    best=min(results,key=results.get)
    words=Counter(t for s in train for t in tokenize(s))
    return {'dev_perplexities':results,'chosen_alpha':best,'test_perplexity':Bigram(train,best).perplexity(test),'bpe_merges':learn_bpe(words,12),'test_documents':len(test)}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Explain why a distribution over successors must sum to one even for an unseen history.

2. Show a tokenizer failure case and propose a task-dependent remedy.

3. Explain what the generated corpus permits you to conclude and what a real-domain evaluation would add.

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Save and submit

From Colab: **Share → General access → Anyone with the link → Viewer**. Click **Copy link**. Paste the link into the Canvas assignment submission.
